# Phase 5: LLM prompting baseline (Hindi NER)

Owner: Ranjeet Kumar Yadav. Plan: [docs/01-roadmap.md](../docs/01-roadmap.md), Phase 5.

Zero-shot and few-shot prompting of a **self-hosted open LLM** (loaded and run in this notebook, no external API) to tag
PER / ORG / LOC in Hindi. The model answers with a JSON array; `scripts/phase5_parser.py` maps it back onto the word
tokens as IOB2 (rules R1 to R9 are written at the top of that file and tested in `scripts/test_phase5_parser.py`).
Scoring uses the **shared** `src.evaluate` (seqeval entity-level micro-F1), the same as every other stage.

**Malformed output is an error, never dropped.** It is scored as all `O` for that sentence and counted.

## Two stages, one notebook

| `STAGE` | Data scored | Purpose | Logs to `results.csv`? |
| --- | --- | --- | --- |
| `"validation"` | a fixed sample of the Hindi **val** split | pick the model, tune the prompt, check parser failures, speed, VRAM | never |
| `"final"` | **all 864** Hindi **gold test** sentences | the frozen model, prompt and parser, scored once per mode | yes (2 rows) |

**Do not tune on gold test.** Gold is only read in the final stage. (Its sentence texts are also loaded in the
validation stage, only to keep them out of the few-shot pool; their labels are not used.)

## How to run (Kaggle)
1. Commit and push this notebook, `scripts/phase5_parser.py` and `scripts/test_phase5_parser.py` to
   `phase5-llm-prompting`, because the run clones that branch and records its commit.
2. Import the notebook on Kaggle with a **GPU T4** and **Internet** on, and attach the `GH_TOKEN` (read-only) Secret.
   Neither candidate model is gated, so no `HF_TOKEN` is needed.
3. **Debug:** `STAGE = "validation"`, `DEBUG = True`, **Save & Run All** (a few minutes). Checks cloning, both model
   loads, generation, parsing and the MuRIL comparison on 16 sentences. Scores are meaningless; keep nothing.
4. **Validation:** `DEBUG = False`, `STAGE = "validation"`, **Save & Run All**. Edit the prompt in section 3 and
   re-run until the validation sample is stable. Record the final prompt's `PROMPT_HASH`.
5. Set `FINAL_MODEL` to the model the selection rule picked, freeze the prompt, and **wait for the lead's go-ahead**
   (Phase 4 must be merged first; both phases write `results.csv`).
6. **Final:** `STAGE = "final"`, `DEBUG = False`, one clean top-to-bottom **Save & Run All**. Download
   `NaamRekha/results/results.csv` and `NaamRekha/results/phase5/` from the Output tab, plus the executed notebook.
   Commit them unchanged.
7. Check the gate locally: `uv run python scripts/phase5_gate.py`.

In [1]:
# Pin versions (docs/04-compute.md). Colab/Kaggle ship their own torch; the rest are pinned for reproducibility.
%pip install -q transformers==5.22.1 accelerate==1.15.0 seqeval==1.2.2 bitsandbytes==0.50.2 sentencepiece==0.2.2  # quietly installs the exact pinned libraries this notebook was validated against, including bitsandbytes for 4-bit model loading

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.


In [2]:
import base64  # used below to base64-encode a temporary git HTTP auth header (git's "basic" auth needs base64)
import json  # used to assemble the auth header value and (later) to serialize/deserialize run artifacts
import os  # used for environment variables, path checks, and filesystem helpers
import subprocess  # used to shell out to `git clone` and `git rev-parse` for reproducibility tracking
import sys  # used to add the repo root (and scripts/) to sys.path so local modules import correctly

REPO_URL = "https://github.com/iPriyadarshi/NaamRekha.git"  # canonical repo URL used to clone the project on Kaggle/Colab
REPO_BRANCH = "phase5-llm-prompting"  # the branch holding this exact notebook, its parser, and its parser tests


def get_secret(name):  # looks up a named secret (e.g. GH_TOKEN) across the three environments this notebook can run in
    # Kaggle Secret, then Colab Secret, then environment variable.
    try:  # attempt the Kaggle-specific secret lookup first
        from kaggle_secrets import UserSecretsClient  # only importable inside a Kaggle kernel
        return UserSecretsClient().get_secret(name)  # fetch the secret from Kaggle's attached Secrets store
    except Exception:  # not on Kaggle, or secret not attached: fall through
        pass  # not on Kaggle (or secret not attached): fall through to the next environment
    try:  # attempt the Colab-specific secret lookup next
        from google.colab import userdata  # only importable inside Google Colab
        return userdata.get(name)  # fetch the secret from Colab's userdata secrets store
    except Exception:  # not on Colab either: fall back to a plain environment variable
        return os.environ.get(name)  # local/plain environment: fall back to a plain env var, or None if unset


# Local: the notebook runs from notebooks/, so the repo root is "..".
# Colab/Kaggle: clone the repo next to the notebook. Check for a repo file, not
# just src/, because Kaggle has its own unrelated /kaggle/src folder.
if os.path.isfile("../src/data.py"):  # running locally from notebooks/: the real src/data.py sits one level up
    REPO_ROOT = os.path.abspath("..")  # repo root is the parent directory
elif os.path.isfile("src/data.py"):  # running with cwd already at the repo root (e.g. repo root opened directly)
    REPO_ROOT = os.path.abspath(".")  # repo root is the current directory
else:  # not running locally with the repo already present: must be Kaggle/Colab, so clone it
    REPO_ROOT = os.path.abspath("NaamRekha")  # not running locally: must be Kaggle/Colab, so clone it into ./NaamRekha next to the notebook
    if not os.path.isdir(REPO_ROOT):  # only clone if it isn't already there (idempotent re-runs)
        env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # copy the environment and disable git's interactive credential prompt
        token = get_secret("GH_TOKEN")  # optional read-only GitHub token, needed only if the repo/branch requires auth
        if token:  # only build the auth header if a token secret was actually found
            # Passed through the environment so it is never saved in .git/config
            # (Kaggle keeps that folder as output) or printed if the clone fails.
            auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()  # build the base64 "user:token" blob HTTP basic auth expects
            env.update(GIT_CONFIG_COUNT="1",  # tells git to apply exactly one extra -c style config entry from the environment
                       GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",  # the config key: an extra HTTP header for github.com requests
                       GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")  # the header value: HTTP Basic auth carrying the token
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_ROOT],  # shallow single-branch clone to save time/bandwidth
                       check=True, env=env)  # raise if the clone fails; use the env with the auth header injected
sys.path.insert(0, REPO_ROOT)  # make the repo root importable so `from src import ...` resolves to the cloned/local copy
## Why add scripts/ to sys.path separately from REPO_ROOT: scripts/ has no __init__.py, so it is not an importable Python package; without this line, `from phase5_parser import ...` below would fail with ModuleNotFoundError even though REPO_ROOT (and therefore `src`) is already importable.
sys.path.insert(0, os.path.join(REPO_ROOT, "scripts"))  # also make scripts/ importable so `from phase5_parser import ...` works

hf_token = get_secret("HF_TOKEN")  # optional Hugging Face token; both candidate LLMs here are ungated/public
if hf_token:  # only set HF_TOKEN if a Hugging Face token secret was actually found
    os.environ["HF_TOKEN"] = hf_token        # optional: only avoids Hub rate limits

GIT_COMMIT = subprocess.check_output(["git", "-C", REPO_ROOT, "rev-parse", "--short", "HEAD"], text=True).strip()  # records the exact commit this run used, for reproducibility logging

from src import LABELS, evaluate, id2label, label2id, load_split, log_result, tag_distribution  # shared project code: labels, scorer, label maps, split loader, results-sheet logger, and a tag-distribution helper
from phase5_parser import format_entities, parse_llm_output, tags_to_entities  # Phase 5's own parser: turns the LLM's JSON answer into IOB2 tags, and the reverse (gold tags into the prompt's JSON answer format)

print("repo:", REPO_ROOT, "| commit:", GIT_COMMIT)  # sanity-check print: confirms which repo copy and commit are in use
print("labels:", LABELS)  # sanity-check print: confirms the label set matches expectations

Cloning into '/kaggle/working/NaamRekha'...


repo: /kaggle/working/NaamRekha | commit: f1d8cbd
labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


## Parser tests (no GPU)

The parser is the riskiest part of this phase, so its tests run first. If any fail, nothing below should be trusted.

In [3]:
proc = subprocess.run([sys.executable, "-X", "utf8", os.path.join(REPO_ROOT, "scripts", "test_phase5_parser.py")],  # build the subprocess command (continued below): run test_phase5_parser.py with the current Python interpreter
                      capture_output=True, text=True, encoding="utf-8")  # run the parser's own test suite as a subprocess, forcing UTF-8 I/O for Devanagari/Bengali text
print(proc.stdout[-400:])  # show the tail of the test output (pass/fail summary) without flooding the notebook
## Why run the parser's tests as a subprocess and hard-assert on exit code, before any GPU work: the parser (not the LLM) is "the riskiest part of this phase" per the markdown above; failing fast here avoids spending GPU-hours generating and scoring against a parser that might have a known bug its own test suite would have caught.
assert proc.returncode == 0, proc.stdout + proc.stderr  # stop immediately if any parser test fails - nothing below this should be trusted otherwise

ntity_object
  PASS  test_span_not_in_sentence_is_unaligned
  PASS  test_tags_are_valid_iob2_for_any_output
  PASS  test_translated_span_is_unaligned
  PASS  test_truncated_json_is_malformed_not_rescued
  PASS  test_unicode_normalisation_nukta
  PASS  test_unknown_label_skipped_and_counted
  PASS  test_wrong_json_shape_is_malformed
  PASS  test_zero_width_joiner_ignored

31/31 parser tests passed



## Config

Every knob is in this one cell. Both candidates are ungated, so nothing needs a token.

**Candidates (at most two, both fit a free T4).** Checked on the Hub: license, gating, exact revision.

| Key | Model | Load | License | Why |
| --- | --- | --- | --- | --- |
| `qwen2.5-3b` | `Qwen/Qwen2.5-3B-Instruct` | fp16 | `qwen-research` (research use only) | small and fast; the earlier draft's choice |
| `qwen2.5-7b-4bit` | `Qwen/Qwen2.5-7B-Instruct` | 4-bit nf4 | Apache-2.0 | larger multilingual model; fits a T4 in 4-bit |

Airavata (Llama 2 license) and Aya Expanse 8B (CC-BY-NC) are gated, and Navarasa uses the Gemma terms, so they were
left out to keep the run reproducible without extra access approvals.

**Selection rule (written before any validation run).** Among candidates whose few-shot malformed rate is at most 5%,
take the highest few-shot validation F1; if several are within 0.01 F1 of the best, take the faster one.

In [4]:
STAGE = "final"     # "validation": model + prompt selection on val, never gold. "final": frozen config on gold.
DEBUG = False             # True: 16 sentences, nothing logged or kept. False: the real runs.

# Kaggle's dual-T4 sessions: keep one GPU so device placement and speed are predictable.
## Why pin to GPU 0 even on Kaggle's dual-T4 sessions: this notebook loads one large model (up to 7B params, 4-bit) and later a second comparison model (MuRIL); keeping everything on a single, predictable device avoids ambiguity about which GPU holds which model's weights when switching between them.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # restrict this process to GPU 0 only

SEED = 42  # global random seed for Python/NumPy/PyTorch, for reproducible shot sampling and generation
LANG = "hi"  # Hindi: the only language prompted in this notebook

CANDIDATES = {  # registry of candidate LLMs this notebook can prompt, keyed by short id
    "qwen2.5-3b": dict(model_id="Qwen/Qwen2.5-3B-Instruct", revision="aa8e72537993ba99e69dfaafa59ed015b17504d1",  # Qwen2.5 3B Instruct, pinned to this exact revision (continued below)
                       load_4bit=False, license="qwen-research"),  # smaller model, loaded in full fp16 precision
    "qwen2.5-7b-4bit": dict(model_id="Qwen/Qwen2.5-7B-Instruct", revision="a09a35458c702b33eeacc393d103063234e8bc28",  # Qwen2.5 7B Instruct, pinned to this exact revision (continued below)
                            load_4bit=True, license="apache-2.0"),  # larger model, loaded 4-bit quantized to fit a free T4's VRAM
}  # both candidates pinned to an exact Hugging Face commit revision, so the weights used can never silently drift
## Why a separate VALIDATION_MODELS list instead of just looping over all of CANDIDATES: round 1 validation did compare both candidates (see the comment above), but having a distinct "which model(s) this run will prompt" knob lets later validation rounds cheaply restrict to just the stronger candidate without touching the final-stage code path below.
VALIDATION_MODELS = ["qwen2.5-7b-4bit"]   # models run in the validation stage. Round 1 ran both (7B won on val F1);
                                          # later tuning rounds run only the chosen one to save time.
FINAL_MODEL = "qwen2.5-7b-4bit"   # FROZEN after validation rounds 1 and 2 (selection rule: higher few-shot val F1)

VAL_RESERVED = 150       # val sentences reserved for validation; never used as few-shot examples
VAL_SAMPLE_SIZE = 150    # how many of them to score (<= VAL_RESERVED; seeded, same every run)
VAL_SAMPLE_SEED = 42  # seed controlling which val sentences are reserved and which of them are sampled
TEST_SIZE = None         # None = all 864 gold sentences (talk to the lead before changing)

PROMPT_VERSION = "v2"    # v1: 8 randomly sampled silver shots. v2: 8 hand-checked shots (SHOT_VAL_IDX).
# Hand-picked few-shot examples: indices into the Hindi val split (read and checked by eye; 7 entity-rich covering
# PER, ORG and LOC, 1 with no entities). Must lie outside the reserved validation sentences. None = seeded sampling.
SHOT_VAL_IDX = [2128, 3126, 7450, 5508, 2314, 7751, 1799, 9703]  # the exact val-split sentence indices used as the frozen few-shot examples
K_SHOT = 8               # in-context examples in few-shot mode (incl. N_EMPTY_SHOTS answered with [])
N_EMPTY_SHOTS = 1  # of the K_SHOT examples, how many are entity-free (answered with an empty list)
SHOT_SEED = 42  # seed used only when SHOT_VAL_IDX is None, to randomly sample the few-shot set instead
BATCH_SIZE = 8  # number of sentences generated together per forward pass
MAX_NEW_TOKENS = 320     # Devanagari costs several tokens per word; an answer cut off mid-JSON counts as malformed
MAX_MALFORMED_RATE = 0.05  # a candidate is disqualified from selection if more than 5% of its answers fail to parse
SELECTION_TIE = 0.01  # candidates within this many F1 points of the best are considered tied, and speed breaks the tie

if DEBUG:  # DEBUG shrinks the validation sample for a fast smoke test
    VAL_SAMPLE_SIZE = 16  # a tiny slice for a fast smoke test
    assert STAGE == "validation", "debug never touches gold"  # hard guard: DEBUG must never accidentally score the frozen gold test set
if STAGE == "final":  # the final stage requires a frozen, already-selected model
## Why assert this here rather than letting a bad FINAL_MODEL fail later inside load_llm(): catching a typo'd/unregistered model key immediately, before any GPU memory is touched, turns a wasted expensive run into an instant, clear error message.
    assert FINAL_MODEL in CANDIDATES, "set FINAL_MODEL to the model chosen on validation"  # catches a typo'd or unregistered model key before any GPU work happens
## Why require BOTH conditions before logging: STAGE == "final" alone isn't enough, because DEBUG=True can also apply inside the final stage and only scores a 16-sentence smoke test (see the DEBUG block above); logging that tiny, meaningless sample into the shared results.csv as if it were the real 864-sentence score would corrupt the shared scoreboard other phases compare against.
LOG_RESULTS = STAGE == "final" and not DEBUG  # only a clean, non-debug final run is allowed to write to the shared results sheet
RUN_KEYS = VALIDATION_MODELS if STAGE == "validation" else [FINAL_MODEL]  # which model(s) this run will actually load and prompt
print("stage:", STAGE, "| debug:", DEBUG, "| models:", RUN_KEYS, "| log results:", LOG_RESULTS)  # sanity-check print of the resolved run configuration

stage: final | debug: False | models: ['qwen2.5-7b-4bit'] | log results: True


In [5]:
import random  # used to seed Python's own RNG and to sample few-shot examples deterministically
from importlib.metadata import PackageNotFoundError, version  # used to record installed package versions for provenance

import numpy as np  # numeric helpers used across the scoring/selection logic
import torch  # the deep learning framework used to load and run the LLM and MuRIL models

random.seed(SEED)  # fix Python's random module so shot sampling is reproducible
np.random.seed(SEED)  # fix NumPy's RNG for any numeric randomness
torch.manual_seed(SEED)  # fix PyTorch's RNG (though generation itself is greedy/deterministic below)

def _ver(pkg):  # safely reads an installed package's version string, even if it isn't installed at all
    try:  # look up this package's installed version
        return version(pkg)  # version string if the package is installed
    except PackageNotFoundError:  # package not installed in this environment
        return "n/a"  # keeps the provenance string from crashing if an optional package is missing

LIB_VERSIONS = (f"torch={torch.__version__}; transformers={_ver('transformers')}; accelerate={_ver('accelerate')}; "  # first half of the provenance string (continued below)
                f"bitsandbytes={_ver('bitsandbytes')}; seqeval={_ver('seqeval')}")  # one compact string logging every library version relevant to reproducing this run

def pick_storage() -> str:  # chooses a writable location for cached raw model outputs, depending on the runtime
    if os.path.isdir("/kaggle/working"):  # running on Kaggle: use its persistent working directory
        return "/kaggle/working/naamrekha"  # Kaggle's persistent output directory for this session
    try:  # attempt to mount Google Drive (only succeeds on Colab)
        from google.colab import drive  # type: ignore  # only importable inside Google Colab
        drive.mount("/content/drive")  # mount the user's Google Drive so caches survive a Colab disconnect
        return "/content/drive/MyDrive/NaamRekha"  # Colab: store caches on the user's mounted Drive so they survive a disconnect
    except ImportError:  # neither Kaggle nor Colab: fall back to a local folder
        return os.path.join(REPO_ROOT, "checkpoints")  # local fallback: a folder inside the repo itself

STORAGE = os.path.join(pick_storage(), "phase5")  # this phase's dedicated cache subfolder
os.makedirs(STORAGE, exist_ok=True)  # ensure the cache directory exists before anything tries to write into it
# Repo-side artifacts (raw outputs, per-sentence predictions, errors). Debug output stays out of results/.
ART_DIR = STORAGE if DEBUG else os.path.join(REPO_ROOT, "results", "phase5")  # DEBUG writes only to the scratch cache; a real run also writes into the tracked results folder
os.makedirs(ART_DIR, exist_ok=True)  # ensure the artifacts directory exists before anything tries to write into it

device = "cuda" if torch.cuda.is_available() else "cpu"  # prefer GPU if one is attached, otherwise fall back to CPU
print(LIB_VERSIONS)  # log every relevant library version for this run's provenance
print("device:", device, "| GPU count:", torch.cuda.device_count(), "| storage:", STORAGE, "| artifacts:", ART_DIR)  # sanity-check print of the runtime environment
if device == "cpu":  # warn early if no GPU is attached, since the model-loading cells below require one
    print("WARNING: no GPU. Parser/data cells work, model cells need a GPU (use Kaggle).")  # the LLM and MuRIL cells below will fail without CUDA

torch=2.11.0+cu130; transformers=5.22.1; accelerate=1.15.0; bitsandbytes=0.50.2; seqeval=1.2.2
device: cuda | GPU count: 1 | storage: /kaggle/working/naamrekha/phase5 | artifacts: /kaggle/working/NaamRekha/results/phase5


## Data

* **Validation stage:** a seeded sample of the Hindi **val** split (silver labels, like Phase 3's size selection).
* **Final stage:** the full Hindi **gold test** set (864 sentences), per [docs/03-conventions.md](../docs/03-conventions.md).
* **Few-shot examples** come from the Hindi **val** split only, from sentences outside the validation sample and
  with no text shared with the gold test set. Prompt `v1` sampled them at random; round 1 showed some silver labels were
  wrong, so `v2` uses 8 **hand-checked** val sentences (`SHOT_VAL_IDX`): 7 entity-rich ones covering PER, ORG and LOC and
  1 with no entities, answered `[]`. Their labels are still silver, a limitation.

In [6]:
def n_ents(tags):  # counts gold entities in a tag sequence by counting "B-" (beginning) tags
    return sum(t.startswith("B-") for t in tags)  # count how many entities (not tokens) this tag sequence contains

val_all = load_split(LANG, "val")  # the full Hindi validation split (silver labels)
test_all = load_split(LANG, "test")  # the full Hindi gold test split (864 sentences)
print(f"val: {len(val_all)} sentences | gold test: {len(test_all)} sentences")  # sanity-check print of split sizes

# The reserved validation sentences are fixed by seed in BOTH stages, so the few-shot pool (and the shots) is the
# same in validation and final. A smaller VAL_SAMPLE_SIZE (debug) is a prefix of the same random draw.
## Why this must hold: VAL_SAMPLE_SIZE sentences are drawn as a prefix of the VAL_RESERVED reserved set below, so a DEBUG run (VAL_SAMPLE_SIZE=16) scores a strict subset of exactly the same sentences a real validation run would use - proving the pipeline works on real data, not on a separately-sampled slice that could behave differently.
assert VAL_SAMPLE_SIZE <= VAL_RESERVED  # the scored sample must fit inside the reserved sentence pool
## Why reserve these sentences by a FIXED seed rather than reserving them freshly each run: both this notebook's validation stage and its final stage (and separate re-runs of either) must agree on exactly which val sentences are "off limits" as few-shot material - a fixed seed makes the reserved set identical across runs without having to persist it anywhere.
reserved = random.Random(VAL_SAMPLE_SEED).sample(range(len(val_all)), VAL_RESERVED)  # a fixed, seeded set of val indices set aside for validation scoring (never used as few-shot material)

# Evaluation rows for this stage.
if STAGE == "validation":  # validation stage: score a reserved, non-gold sample of the val split
    val_idx = sorted(reserved[:VAL_SAMPLE_SIZE])  # a fixed prefix of the reserved sentences, so DEBUG is a strict subset of a real validation run
    eval_rows = [val_all[i] for i in val_idx]  # the actual (words, tags) rows being scored this run
    eval_name = f"val sample {len(eval_rows)}/{len(val_all)} (seed {VAL_SAMPLE_SEED})"  # human-readable label for prints below
else:  # final stage: score the full, frozen gold test set
    val_idx = []  # not used in the final stage
    eval_rows = test_all if TEST_SIZE is None else random.Random(VAL_SAMPLE_SEED).sample(test_all, TEST_SIZE)  # normally every gold test sentence; TEST_SIZE lets a reduced run be sampled if ever needed
    eval_name = f"gold test {len(eval_rows)}/{len(test_all)}"  # human-readable label for prints below
print("evaluating on:", eval_name)  # confirms exactly what this run is about to score
d = tag_distribution(eval_rows); tot = sum(d.values())  # count of each tag across the evaluation rows, and their total
print({k: f"{v} ({v / tot:.1%})" for k, v in sorted(d.items())})  # label distribution, as both raw counts and percentages

# Few-shot pool: val only, outside the reserved validation sentences and with no text shared with gold test.
fixed_val_idx = set(reserved)  # fast membership checks for "is this val sentence reserved for validation scoring"
test_texts = {tuple(w) for w, _ in test_all}  # every gold test sentence's exact word sequence, to guard against test-set leakage into the few-shot pool
pool = [(i, r) for i, r in enumerate(val_all)  # candidate few-shot sentences (continued below): must be outside the reserved sentences, length-filtered, and clean
        if i not in fixed_val_idx and tuple(r[0]) not in test_texts and 6 <= len(r[0]) <= 25  # (continued) not reserved, not in the gold test set, and a reasonable sentence length
        and not any("\\" in x for x in r[0])]        # skip rows with literal escape junk such as "\xa0"
rich = [r for _, r in pool if n_ents(r[1]) >= 2]  # candidate shots with at least 2 entities, to demonstrate the task clearly
empty = [r for _, r in pool if n_ents(r[1]) == 0]  # candidate shots with no entities at all, to demonstrate the empty-answer case
print(f"few-shot pool: {len(rich)} entity-rich, {len(empty)} entity-free")  # sanity-check print of how much shot material is available

def pick_shots(k, n_empty, seed):  # randomly builds a k-shot set covering all three entity types, deterministically
    for attempt in range(1000):                                  # deterministic: first seed that covers all types
        r = random.Random(seed + attempt)  # a fresh, seeded RNG per attempt, so the result is fully reproducible
        chosen = r.sample(rich, k - n_empty)  # sample the entity-rich shots first
## Why require ALL THREE entity types to appear across the sampled shots: a few-shot set that happened to omit, say, ORG examples would never demonstrate what an ORG answer looks like, biasing the model toward under-predicting (or mis-typing) organisations it was never shown in context.
        if {t[2:] for _, tags in chosen for t in tags if t.startswith("B-")} == {"PER", "ORG", "LOC"}:  # require this particular sample to cover all three entity types between it
            shots = chosen + r.sample(empty, n_empty)  # top up with the required number of entity-free examples
            r.shuffle(shots)  # mix entity-rich and entity-free shots so their order isn't a giveaway
            return shots  # a valid shot set covering PER, ORG and LOC was found
    raise RuntimeError("no shot set covers PER, ORG and LOC")  # extremely unlikely after 1000 attempts; signals a data problem if it happens

## Why freeze specific hand-checked indices (prompt v2) instead of always using pick_shots' seeded random sampling (prompt v1): per the Data section above, round 1's randomly-sampled shots could land on a sentence whose SILVER (model-generated, unverified) label was itself wrong, teaching the few-shot prompt an incorrect example; hand-checking the frozen v2 shots removes that specific risk for the version actually used in the final run.
if SHOT_VAL_IDX is not None:  # prompt v2: use the frozen, hand-checked shot indices
    assert len(SHOT_VAL_IDX) == K_SHOT and not set(SHOT_VAL_IDX) & set(reserved), "shots must be outside the reserved sentences"  # the frozen hand-picked shots must not leak into the validation-scoring sentences
    shots = [val_all[i] for i in SHOT_VAL_IDX]  # use the frozen, hand-checked few-shot examples (prompt v2)
else:  # prompt v1: fall back to seeded random shot sampling
    shots = pick_shots(K_SHOT, N_EMPTY_SHOTS, SHOT_SEED)  # fall back to seeded random sampling (prompt v1 behavior)
eval_texts = {tuple(w) for w, _ in eval_rows}  # the exact word sequences being evaluated this run, to guard against shot/eval overlap
## Why two separate overlap asserts (this one and the eval_texts one below) instead of one combined check: they guard against two DIFFERENT leaks - this one stops a shot from being a sentence from the frozen gold test set (test contamination that would inflate scores generally), while the next one stops a shot from being one of THIS run's own evaluation sentences (which would hand the model its own answer key for that specific sentence).
assert not any(tuple(w) in test_texts for w, _ in shots), "few-shot example overlaps the gold test set"  # a shot that is itself a gold test sentence would leak test data into the prompt
assert not any(tuple(w) in eval_texts for w, _ in shots), "few-shot example overlaps the evaluation sentences"  # a shot that is itself a sentence being scored would make that sentence trivially easy
for w, t in shots:  # print every chosen few-shot example for manual inspection
    print("  shot:", " ".join(w), "->", format_entities(tags_to_entities(w, t)))  # print every few-shot example and its gold answer, for manual inspection

data/hi_IndicNER_v1.0.zip: reconstructing file:   0%|          |  0.00B / 82.3MB            

data/hi_IndicNER_v1.0.zip: downloading bytes:           |  0.00B            

val: 13460 sentences | gold test: 864 sentences
evaluating on: gold test 864/864
{'B-LOC': '613 (3.1%)', 'B-ORG': '521 (2.6%)', 'B-PER': '788 (4.0%)', 'I-LOC': '199 (1.0%)', 'I-ORG': '512 (2.6%)', 'I-PER': '747 (3.8%)', 'O': '16513 (83.0%)'}
few-shot pool: 3703 entity-rich, 912 entity-free
  shot: दोबे ने कहा कि कश्मीर की स्थिति को लेकर ओआईसी चिंतित है । -> [{"text": "दोबे", "type": "PER"}, {"text": "कश्मीर", "type": "LOC"}, {"text": "ओआईसी", "type": "ORG"}]
  shot: पूर्व केंद्रीय मंत्री और मणिपुर की राज्यपाल नजमा हेपतुल्ला को जामिया मिलिया इस्लामिया की चांसलर नियुक्त किया गया है . -> [{"text": "मणिपुर", "type": "LOC"}, {"text": "नजमा हेपतुल्ला", "type": "PER"}, {"text": "जामिया मिलिया इस्लामिया", "type": "ORG"}]
  shot: एमक्यूएम के 24 सांसदों ने अब्बासी को समर्थन दिया । -> [{"text": "एमक्यूएम", "type": "ORG"}, {"text": "अब्बासी", "type": "PER"}]
  shot: निर्वाचन आयोग ने आज उत्तर प्रदेश में राज्यसभा की एक सीट के लिए उपचुनाव की अधिसूचना जारी कर दी । -> [{"text": "निर्वाचन आयोग", "type":

## Prompt

One fixed answer format, shared by both modes: `[{"text": "<entity as written>", "type": "PER|ORG|LOC"}]`.
Zero-shot is the instructions alone; few-shot is the same instructions plus the fixed shots above as chat turns.
`PROMPT_HASH` fingerprints the instructions and the shots so a run can be matched to the prompt that produced it.
**Edit the prompt only in the validation stage**, then freeze it.

In [7]:
import hashlib  # used to fingerprint the prompt text and the parser's own source code

SYSTEM = (  # the fixed system prompt, built from several concatenated string literals (continued below)
    "You are a Named Entity Recognition system for Hindi text. "  # states the task and the language
    "Find every person (PER), organisation (ORG) and location (LOC) name in the sentence. "  # states exactly which three entity types to find
    "PER: names of people. ORG: companies, political parties, institutions, government bodies, teams. "  # defines what counts as PER and ORG
    "LOC: countries, states, cities, rivers and other places. "  # defines what counts as LOC
## Why this instruction matters mechanically, not just stylistically: phase5_parser.parse_llm_output aligns the model's returned entity text back onto the ORIGINAL word tokens by matching the text itself; if the model paraphrased, transliterated or translated an entity, that string match would fail and the entity would be counted as `unaligned` even though the model "found" it correctly in spirit.
    "Copy each entity exactly as it is written in the sentence: same script, same spelling, no translation. "  # forbids transliteration/translation, so the parser can align the answer back onto the original words
## Why forbid any explanation text: generation is capped at MAX_NEW_TOKENS; every token spent on prose before/after the JSON is a token not available to finish the actual answer, raising the risk of hitting the cap mid-JSON (counted as `truncated`) and giving the parser extra, unwanted text to strip out before it can even find the array.
    "Reply with ONLY a JSON array and no explanation, in this format: "  # forces a strict, parseable answer format with no extra prose
    '[{"text": "<entity as written>", "type": "PER"}, {"text": "<entity as written>", "type": "LOC"}]. '  # shows the exact JSON shape the model must produce
    "Use the type PER, ORG or LOC. If the sentence has no entities, reply []."  # closes the instructions: the allowed type values, and the empty-answer convention
)  # the fixed system prompt: task instructions, entity type definitions, and the exact required answer format

def user_msg(words):  # formats one sentence as the user turn of the chat prompt
    return "Sentence: " + " ".join(words)  # re-joins the word list into a plain sentence string for the prompt

def build_messages(words, shots=()):  # assembles the full chat-formatted prompt: system instructions, optional few-shot turns, then the real sentence
    msgs = [{"role": "system", "content": SYSTEM}]  # start with the fixed task instructions
    for w, t in shots:  # add one user/assistant turn pair per few-shot example
        msgs += [{"role": "user", "content": user_msg(w)},  # the shot's sentence as a user turn (continued below)
                 {"role": "assistant", "content": format_entities(tags_to_entities(w, t))}]  # one example sentence plus its gold-answer JSON, as a user/assistant turn pair
    msgs.append({"role": "user", "content": user_msg(words)})  # finally, the real sentence to be tagged
    return msgs  # ready to be passed to the tokenizer's chat template

PARSER_SHA = hashlib.sha1(open(os.path.join(REPO_ROOT, "scripts", "phase5_parser.py"), "rb").read()).hexdigest()[:8]  # short hash of the parser's source, so logged results can be traced to the exact parsing logic used
## Why hash the rendered shot messages together with SYSTEM, instead of hashing SYSTEM (the instructions) alone: the actual prompt sent to the model also depends on WHICH few-shot examples are included; if only the instructions text were hashed, two runs using different shot sets (e.g. v1 random vs v2 hand-checked) would get the identical PROMPT_HASH despite producing different model inputs.
PROMPT_HASH = hashlib.sha1((PROMPT_VERSION + SYSTEM + json.dumps(build_messages([""], shots), ensure_ascii=False))  # hash the prompt version, instructions, and rendered shot messages together (continued below)
                           .encode("utf-8")).hexdigest()[:8]  # short hash fingerprinting the exact prompt version, instructions, and shot set used
print(f"prompt {PROMPT_VERSION}/{PROMPT_HASH} | parser {PARSER_SHA}")  # confirms which prompt and parser versions this run used
print(json.dumps(build_messages(eval_rows[0][0], shots[:1]), ensure_ascii=False, indent=1)[:1500])  # print a truncated example prompt (system + one shot + the first eval sentence) for manual inspection

prompt v2/85343238 | parser ffb1d6e7
[
 {
  "role": "system",
  "content": "You are a Named Entity Recognition system for Hindi text. Find every person (PER), organisation (ORG) and location (LOC) name in the sentence. PER: names of people. ORG: companies, political parties, institutions, government bodies, teams. LOC: countries, states, cities, rivers and other places. Copy each entity exactly as it is written in the sentence: same script, same spelling, no translation. Reply with ONLY a JSON array and no explanation, in this format: [{\"text\": \"<entity as written>\", \"type\": \"PER\"}, {\"text\": \"<entity as written>\", \"type\": \"LOC\"}]. Use the type PER, ORG or LOC. If the sentence has no entities, reply []."
 },
 {
  "role": "user",
  "content": "Sentence: दोबे ने कहा कि कश्मीर की स्थिति को लेकर ओआईसी चिंतित है ।"
 },
 {
  "role": "assistant",
  "content": "[{\"text\": \"दोबे\", \"type\": \"PER\"}, {\"text\": \"कश्मीर\", \"type\": \"LOC\"}, {\"text\": \"ओआईसी\", \"type\": \"

In [8]:
## Why round-trip the GOLD tags through format_entities -> parse_llm_output instead of just trusting the parser: this establishes an upper bound on achievable F1 given the parser's own alignment rules (e.g. an entity text that appears twice in a sentence but is tagged only once in gold); any CEILING below 1.0 proves that specific amount of lost F1 is a property of the evaluation harness, not of the LLM's answers.
# Parser ceiling: feed the GOLD answers through the parser. Anything below 1.0 is lost by the parser rules
# (e.g. a name tagged once in gold but mentioned twice), not by the model.
ceiling_pred = [parse_llm_output(format_entities(tags_to_entities(w, t)), w).tags for w, t in eval_rows]  # round-trip each sentence's gold tags through format->parse, simulating a "perfect" model
CEILING = evaluate([t for _, t in eval_rows], ceiling_pred)  # the best F1 achievable given the parser's own rules, independent of model quality
print("parser ceiling on", eval_name, ":", {k: round(v, 4) for k, v in CEILING.items()})  # prints the ceiling score, for comparison against the actual LLM results below

parser ceiling on gold test 864/864 : {'F1_overall': 0.9984, 'F1_PER': 0.9994, 'F1_ORG': 0.9971, 'F1_LOC': 0.9984}


## Model loading, generation and scoring

Each candidate is loaded from its **pinned revision**, run in both modes, then freed (`del` +
`torch.cuda.empty_cache()`) before the next model, so only one large model is in VRAM at a time.
Greedy decoding. Raw outputs are cached under `STORAGE` so a disconnect does not lose generated text.

In [9]:
import gc  # used to force garbage collection when freeing GPU memory between model loads
import time  # used to measure generation wall-clock time for the speed/VRAM comparison table
from collections import Counter  # used to tally malformed-output reasons
from tqdm.auto import tqdm  # progress bar for the batched generation loop

def free_gpu():  # releases Python objects and clears the CUDA cache so the next model has room on the GPU
    gc.collect()  # force Python garbage collection of just-deleted model objects
    if torch.cuda.is_available():  # only clearing a CUDA cache makes sense if there is a GPU in the first place
        torch.cuda.empty_cache()  # return freed GPU memory to the CUDA allocator's free pool

def load_llm(cfg):  # loads one candidate LLM and its tokenizer at the pinned revision, optionally 4-bit quantized
    if not torch.cuda.is_available():  # refuse to proceed with a model that needs a GPU when none is attached
        raise RuntimeError("No GPU attached. Kaggle: Settings > Accelerator > GPU T4. Colab: Runtime > T4 GPU.")  # fail with an actionable message rather than crashing deep inside model loading
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig  # imported lazily here since they're only needed once a GPU is confirmed present
## Why padding_side="left" instead of the more common right-padding: in batched generation with a decoder-only model, the NEXT token is always produced at the sequence's rightmost position; right-padding would place pad tokens after some sequences' real content, corrupting exactly the position generation continues from, whereas left-padding keeps every sequence's real content flush against that generation point.
    tok = AutoTokenizer.from_pretrained(cfg["model_id"], revision=cfg["revision"], padding_side="left")  # left-padding is required for correct batched generation with a decoder-only model
    if tok.pad_token is None:  # this model's tokenizer has no dedicated padding token defined
        tok.pad_token = tok.eos_token  # some causal LMs ship without a dedicated pad token; reuse end-of-sequence as a safe stand-in
    kw = dict(revision=cfg["revision"], device_map={"": 0})  # pin the exact checkpoint revision and place the whole model on GPU 0
    if cfg["load_4bit"]:  # this candidate should be loaded quantized to fit a free T4's VRAM
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",  # 4-bit NF4 quantization config (continued below): computation still happens in fp16
                                                       bnb_4bit_compute_dtype=torch.float16)  # 4-bit NF4 quantization, computing in fp16, to fit the larger model on a free T4
    else:  # the smaller candidate fits comfortably without quantization
        kw["dtype"] = torch.float16  # the smaller candidate is loaded in full fp16 precision instead
    model = AutoModelForCausalLM.from_pretrained(cfg["model_id"], **kw).eval()  # load the weights and switch to inference mode
    return tok, model  # ready for prompting

@torch.no_grad()  # disables gradient tracking for the whole function: this is inference only
def generate(tok, model, rows, shots, cache_path):  # runs greedy generation over every row, with on-disk caching
    """Greedy answers for every row. Returns (raw texts, truncated flags, seconds spent generating)."""  # function docstring describing its return value
## Why cache raw generations keyed on cache_path (which itself encodes stage/model/mode/prompt-hash/sentence-count): Kaggle/Colab GPU sessions can disconnect mid-run, and regenerating is the expensive part of this notebook; encoding every setting that could change the output into the filename (rather than a fixed name) prevents a stale cache from a DIFFERENT config being silently reused as if it were current.
    if os.path.exists(cache_path):  # reuse a previous run's raw outputs if they're already on disk
        cached = [json.loads(l) for l in open(cache_path, encoding="utf-8")]  # one JSON record per cached sentence
        if len(cached) == len(rows):  # only trust the cache if it covers exactly this many sentences
            print(f"  loaded {len(cached)} cached outputs from {cache_path}")  # confirms the cache was reused instead of regenerating
            return [c["raw"] for c in cached], [c["truncated"] for c in cached], None  # None seconds: no generation actually happened this run
    stop = set(np.atleast_1d(model.generation_config.eos_token_id).tolist()) | {tok.pad_token_id}  # every token id that should be treated as "the model stopped naturally"
## Why restore original order via raw[i], trunc[i] = ... (indexed by i) instead of just returning results in the sorted order they were generated in: sorting by length is purely an efficiency trick (less wasted padding per batch); callers of generate() expect outputs aligned with `rows` in its original order, so the pre-allocated, index-written arrays undo the reordering before returning.
    order = sorted(range(len(rows)), key=lambda i: len(rows[i][0]))        # similar lengths per batch = less padding
    raw, trunc = [None] * len(rows), [False] * len(rows)  # pre-allocated output slots, filled in original order below
    t0 = time.time()  # start the generation timer
    for b in tqdm(range(0, len(order), BATCH_SIZE), desc="generate"):  # iterate length-sorted batches, with a progress bar
        ids = order[b:b + BATCH_SIZE]  # original sentence indices in this batch
        prompts = [tok.apply_chat_template(build_messages(rows[i][0], shots), tokenize=False, add_generation_prompt=True)  # render each sentence's chat prompt as text (continued below)
                   for i in ids]  # render each sentence's full chat prompt as a single text string, ready for the model
        enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)  # tokenize and left-pad the batch, moved to the model's device
## Why do_sample=False (greedy) rather than sampling: the final stage is meant to be run once and frozen; a stochastic decoding strategy would make F1 and the malformed-rate not exactly reproducible between reruns, undermining the "one clean top-to-bottom Save & Run All" frozen-result philosophy stated in the instructions above.
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tok.pad_token_id)  # greedy (do_sample=False) decoding for deterministic, reproducible answers
        gen = out[:, enc["input_ids"].shape[1]:]  # strip the prompt tokens off, keeping only the newly generated continuation
        for i, row, text in zip(ids, gen.tolist(), tok.batch_decode(gen, skip_special_tokens=True)):  # pair each sentence's original index with its generated token ids and decoded text
            raw[i], trunc[i] = text, not any(t in stop for t in row)       # no stop token = ran out of tokens
    secs = time.time() - t0  # total wall-clock time spent generating this batch of sentences
    with open(cache_path, "w", encoding="utf-8") as f:  # persist the raw outputs so a later run can reuse them instead of regenerating
        for r, t in zip(raw, trunc):  # one cache line per sentence: its raw text and whether it was truncated
            f.write(json.dumps({"raw": r, "truncated": t}, ensure_ascii=False) + "\n")  # cache the raw text so a disconnect/rerun doesn't need to regenerate it
    return raw, trunc, secs  # raw model text, truncation flags, and total generation time for this run

def score(rows, raw, truncated):  # parses every raw LLM answer back into tags and scores them against gold
    parsed = [parse_llm_output(r, w) for r, (w, _) in zip(raw, rows)]  # run the Phase 5 parser on each sentence's raw output
    gold = [t for _, t in rows]  # the gold tag sequences being compared against
    scores = evaluate(gold, [p.tags for p in parsed])  # seqeval entity-level F1 etc., using the parser's recovered tags
    stats = dict(  # diagnostics dict summarizing how well the model's answers parsed (continued below)
        n=len(rows),  # total sentences scored
        malformed=sum(p.malformed for p in parsed),  # count of answers the parser could not interpret at all (scored as all-O)
        malformed_reasons=dict(Counter(p.reason for p in parsed if p.malformed)),  # breakdown of why each malformed answer failed to parse
        truncated=sum(truncated),  # count of answers that hit MAX_NEW_TOKENS before finishing
        noisy=sum(p.noisy for p in parsed),  # count of answers with extra prose/code fences around the JSON
        invalid_items=sum(p.n_invalid_items for p in parsed),  # count of JSON array items the parser had to discard as malformed
        unknown_label=sum(p.n_unknown_label for p in parsed),  # count of entities tagged with a type outside PER/ORG/LOC
        unaligned=sum(p.n_unaligned for p in parsed),  # count of entity texts the parser could not align back onto the sentence's words
        overlap_dropped=sum(p.n_overlap_dropped for p in parsed),  # count of entities dropped because they overlapped another already-placed entity
        conflict=sum(p.n_conflict for p in parsed),  # count of conflicting entity assignments the parser had to resolve
        items=sum(p.n_items for p in parsed),  # total entities the model proposed, valid or not
    )  # closes the stats dict literal
    return scores, stats, parsed  # overall metrics, a diagnostics dict, and the per-sentence parsed results

In [10]:
results = {}     # (model key, mode) -> dict
MODES = [("zero-shot", ()), ("few-shot", tuple(shots))]  # the two prompting modes compared for every candidate model

for key in RUN_KEYS:  # RUN_KEYS is either every validation model, or just the one frozen FINAL_MODEL
    cfg = CANDIDATES[key]  # this candidate's model id, revision, and loading config
    print(f"\n=== {key}: {cfg['model_id']} @ {cfg['revision'][:8]} | 4bit={cfg['load_4bit']} | license={cfg['license']}")  # banner identifying exactly which checkpoint is about to run
    free_gpu()  # make sure no memory from a previous model lingers before loading this one
    tok, model = load_llm(cfg)  # load this candidate's tokenizer and weights
## Why a separate fingerprint on top of the already-distinct PROMPT_HASH and PARSER_SHA: PROMPT_HASH alone does not capture generation settings like BATCH_SIZE/MAX_NEW_TOKENS or which exact model/revision/quantization was used; this single combined hash, stored in results.csv's notes, lets two logged rows be checked for "run under truly identical settings" without manually comparing a dozen separate fields.
    fingerprint = hashlib.sha1(json.dumps([cfg["model_id"], cfg["revision"], cfg["load_4bit"], PROMPT_HASH, PARSER_SHA,  # hash together every setting that could change this run's output (continued below)
                                           K_SHOT, N_EMPTY_SHOTS, SHOT_SEED, BATCH_SIZE, MAX_NEW_TOKENS]).encode()  # (continued) the remaining settings folded into the fingerprint
                               ).hexdigest()[:8]  # a single hash summarizing every setting that could affect this run's output, for the results-sheet notes
    print("loaded | VRAM after load (GB):", round(torch.cuda.memory_allocated() / 1e9, 2), "| config fingerprint:", fingerprint)  # confirms the model loaded and reports its memory footprint
    for mode, sh in MODES:  # zero-shot (no in-context examples) and few-shot (with the frozen shot set)
        torch.cuda.reset_peak_memory_stats()  # reset peak-memory tracking so this mode's peak VRAM is measured in isolation
        cache = os.path.join(STORAGE, f"{STAGE}_{key}_{mode}_{PROMPT_HASH}_{len(eval_rows)}.jsonl")  # cache path keyed by every setting that would change the output, so stale caches are never reused by accident
        raw, trunc, secs = generate(tok, model, eval_rows, sh, cache)  # generate (or load cached) raw answers for every evaluation sentence
        scores, stats, parsed = score(eval_rows, raw, trunc)  # parse the raw answers and score them against gold
        results[(key, mode)] = dict(cfg=cfg, scores=scores, stats=stats, parsed=parsed, raw=raw, truncated=trunc,  # store everything needed later for summaries/selection/error-analysis (continued below)
                                    sec_per_sent=None if secs is None else secs / len(eval_rows),  # (continued) average generation time per sentence, if this wasn't a cache hit
                                    peak_gb=torch.cuda.max_memory_allocated() / 1e9, fingerprint=fingerprint)  # everything needed later for summaries, selection, and error analysis
        print(f"[{key} / {mode}] F1 overall {scores['F1_overall']:.4f} | PER {scores['F1_PER']:.4f} | "  # headline accuracy numbers for this model/mode (continued below)
              f"ORG {scores['F1_ORG']:.4f} | LOC {scores['F1_LOC']:.4f}")  # headline metrics for this model/mode combination
        print(f"   malformed {stats['malformed']}/{stats['n']} {stats['malformed_reasons']} | truncated {stats['truncated']} | "  # failure-mode diagnostics for this model/mode (continued below)
              f"unaligned spans {stats['unaligned']} | peak VRAM {results[(key, mode)]['peak_gb']:.1f} GB")  # diagnostics: how often the model's output was unusable, and how much memory it used
    del model, tok  # drop references so this candidate's memory can be freed before the next one loads
    free_gpu()  # clear GPU memory before moving to the next candidate (or finishing)


=== qwen2.5-7b-4bit: Qwen/Qwen2.5-7B-Instruct @ a09a3545 | 4bit=True | license=apache-2.0


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (6 outputs left out)

generate:   0%|          | 0/108 [00:00<?, ?it/s]

[qwen2.5-7b-4bit / zero-shot] F1 overall 0.6328 | PER 0.6397 | ORG 0.5644 | LOC 0.6812
   malformed 11/864 {'broken_json': 11} | truncated 6 | unaligned spans 32 | peak VRAM 7.4 GB


generate:   0%|          | 0/108 [00:00<?, ?it/s]

[qwen2.5-7b-4bit / few-shot] F1 overall 0.6643 | PER 0.6714 | ORG 0.5956 | LOC 0.7190
   malformed 5/864 {'broken_json': 5} | truncated 5 | unaligned spans 31 | peak VRAM 13.4 GB


## Summary and model selection

Speed and VRAM matter for the final run: 864 sentences in each of two modes has to fit one Kaggle session.

In [11]:
import pandas as pd  # tabular summary table for comparing every (model, mode) run
try:  # IPython's rich display is only available inside a notebook kernel
    from IPython.display import display  # pretty-prints a DataFrame in a notebook cell's output
except ImportError:  # running as a plain script: fall back to plain print
    display = print  # fall back to plain print if IPython's display isn't available (e.g. running as a plain script)

rows_ = []  # will become the `summary` DataFrame comparing every candidate/mode run
for (key, mode), r in results.items():  # build one summary row per model/mode run
    s, st = r["scores"], r["stats"]  # this run's metrics and diagnostics dicts
    sec = r["sec_per_sent"]  # this run's average generation time per sentence (None for a cache hit)
    rows_.append({"model": key, "mode": mode, "F1": round(s["F1_overall"], 4), "PER": round(s["F1_PER"], 4),  # start this summary row (continued below): identifying columns and per-type F1
                  "ORG": round(s["F1_ORG"], 4), "LOC": round(s["F1_LOC"], 4),  # (continued) remaining per-type F1 scores
                  "malformed": f"{st['malformed']}/{st['n']}", "truncated": st["truncated"], "unaligned": st["unaligned"],  # (continued) failure-mode counts
                  "unknown_label": st["unknown_label"], "sec/sent": None if sec is None else round(sec, 2),  # (continued) more failure-mode counts and timing
                  "est. min for 864": None if sec is None else round(sec * 864 / 60, 1),  # (continued) extrapolated time to run the full 864-sentence gold set
                  "peak VRAM GB": round(r["peak_gb"], 1)})  # one row summarizing accuracy, failure modes, and cost for this model/mode
summary = pd.DataFrame(rows_)  # the full head-to-head comparison table, ready to display
display(summary)  # the full head-to-head comparison table
print(f"parser ceiling {CEILING['F1_overall']:.4f} | evaluated on {eval_name}")  # reminder of the maximum F1 the parser itself allows, for context

def pick_model():  # implements the pre-registered selection rule: highest few-shot F1 among candidates under the malformed-rate limit, ties broken by speed
## Why filter on malformed rate BEFORE comparing F1 at all: a candidate could reach a deceptively high F1 by having many of its hardest sentences fail to parse (scored as all-O) while its small set of parseable answers happen to be easy ones; excluding high-malformed-rate candidates up front stops a broken-but-lucky model from winning the selection rule.
    few = {k: r for (k, m), r in results.items() if m == "few-shot" and r["stats"]["malformed"] / r["stats"]["n"] <= MAX_MALFORMED_RATE}  # only consider few-shot runs that produced usably few malformed answers
    if not few:  # no candidate's few-shot run met the malformed-rate limit at all
        return None  # no candidate qualifies at all
    best = max(r["scores"]["F1_overall"] for r in few.values())  # the best qualifying F1 score
## Why break near-ties by speed (sec_per_sent) instead of by raw F1 alone: the selection rule was written BEFORE any validation run specifically to avoid picking a model for a statistically meaningless F1 difference; among genuinely-tied candidates, generation speed is the only remaining criterion that matters for actually completing the 864-sentence final run in one Kaggle session.
    close = [k for k, r in few.items() if r["scores"]["F1_overall"] >= best - SELECTION_TIE]  # every candidate within SELECTION_TIE of the best, i.e. "tied"
    return min(close, key=lambda k: results[(k, "few-shot")]["sec_per_sent"] or 0)  # among tied candidates, prefer the fastest

if STAGE == "validation":  # model selection only applies during validation; the final stage already has a frozen model
    chosen = pick_model()  # only meaningful during validation; the final stage runs a single, already-chosen model
    print("selection rule picks:", chosen, "(set FINAL_MODEL to this, then freeze the prompt)" if chosen  # tells the operator what to do next, whether or not a model qualified (continued below)
          else "-> no candidate met the malformed-output limit; fix the prompt first")  # tells the notebook's operator exactly what to do next
    if DEBUG:  # a DEBUG run's numbers are meaningless and shouldn't inform model selection
        print("DEBUG run: numbers above only prove the pipeline works.")  # a DEBUG run's F1/selection numbers are meaningless and shouldn't be acted on

,model,mode,F1,PER,ORG,LOC,malformed,truncated,unaligned,unknown_label,sec/sent,est. min for 864,peak VRAM GB
0,qwen2.5-7b-4bit,zero-shot,0.6328,0.6397,0.5644,0.6812,11/864,6,32,1,2.65,38.1,7.4
1,qwen2.5-7b-4bit,few-shot,0.6643,0.6714,0.5956,0.7190,5/864,5,31,1,5.61,80.7,13.4


parser ceiling 0.9984 | evaluated on gold test 864/864


## Log to the shared results sheet (final stage only)

One row per mode. `checkpoint_path` is the model's pinned Hugging Face URL. The row is printed exactly as written so
the gate can match it to this notebook's output.

In [12]:
def notes_for(key, mode, r):  # builds the packed "key=value; ..." notes string logged alongside this run's score
    st, cfg = r["stats"], r["cfg"]  # this run's diagnostics dict and model config
    return (f"phase=5; mode={mode}; k_shot={K_SHOT if mode == 'few-shot' else 0}; test_n={st['n']}/{len(test_all)}; "  # start of the packed notes string (continued below): phase, mode, shot count, sentence count
            f"sample_seed=n/a (full test); seed={SEED}; shot_seed={SHOT_SEED}; model={cfg['model_id']}; "  # (continued) seeds and which model was used
            f"revision={cfg['revision']}; load_4bit={cfg['load_4bit']}; license={cfg['license']}; greedy; "  # (continued) exact pinned revision, quantization, license, and decoding strategy
            f"batch={BATCH_SIZE}; max_new_tokens={MAX_NEW_TOKENS}; prompt={PROMPT_VERSION}/{PROMPT_HASH}; "  # (continued) generation settings and the prompt fingerprint
            f"parser={PARSER_SHA}; config={r['fingerprint']}; malformed={st['malformed']}; "  # (continued) parser version, overall config fingerprint, and malformed count
            f"malformed_rate={st['malformed'] / st['n']:.4f}; truncated={st['truncated']}; noisy={st['noisy']}; "  # (continued) more failure-mode rates
            f"invalid_items={st['invalid_items']}; unknown_label={st['unknown_label']}; unaligned_spans={st['unaligned']}; "  # (continued) even more failure-mode counts
            f"git={GIT_COMMIT}; {LIB_VERSIONS}")  # every setting and diagnostic needed to fully reproduce or audit this logged score later

if LOG_RESULTS:  # only true in a clean, non-DEBUG final-stage run
## Why both conditions, not just one: len(RUN_KEYS) == 1 alone wouldn't catch a run accidentally scoring a reduced TEST_SIZE slice as if it were official; len(eval_rows) == len(test_all) alone wouldn't catch a validation-stage sweep over multiple candidate models being logged as if each were "the" final result - together they guarantee the row appended to results.csv really is one single, complete, official run.
    assert len(RUN_KEYS) == 1 and len(eval_rows) == len(test_all), "final run = one model, all gold sentences"  # guards against accidentally logging a partial or multi-model run as if it were the official final result
    from src import RESULTS_PATH  # the shared scoreboard's file path
    for (key, mode), r in results.items():  # log one row per (model, mode) run that was actually produced this session
        cfg = r["cfg"]  # this run's model config, for the checkpoint URL below
        log_result(model=f"LLM-{mode} ({cfg['model_id'].split('/')[-1]})", train_lang="none", test_lang=LANG,  # append this run's row to the shared results sheet (continued below)
                   subset_size=0, epochs=0, scores=r["scores"],  # (continued) no training subset/epochs since this is prompting, not fine-tuning
                   checkpoint_path=f"https://huggingface.co/{cfg['model_id']}/tree/{cfg['revision']}",  # (continued) the pinned Hugging Face URL identifying the exact model weights used
                   notes=notes_for(key, mode, r))  # append one row per mode (zero-shot, few-shot) to the shared results.csv
    with open(RESULTS_PATH, encoding="utf-8") as f:  # re-read the results sheet to print back exactly what was just logged
        for line in f:  # scan every row in the sheet
            if line.startswith("LLM-"):  # only print the LLM rows just appended by this run
                print(line.rstrip("\n"))                  # exactly the logged rows
else:  # not a clean final-stage run: nothing is logged
    print("not the final stage (or DEBUG): nothing written to results.csv")  # confirms the shared scoreboard was left untouched

LLM-zero-shot (Qwen2.5-7B-Instruct),none,hi,0,0,0.6396562308164517,0.5644283121597096,0.6812030075187969,0.632849051957646,https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/tree/a09a35458c702b33eeacc393d103063234e8bc28,phase=5; mode=zero-shot; k_shot=0; test_n=864/864; sample_seed=n/a (full test); seed=42; shot_seed=42; model=Qwen/Qwen2.5-7B-Instruct; revision=a09a35458c702b33eeacc393d103063234e8bc28; load_4bit=True; license=apache-2.0; greedy; batch=8; max_new_tokens=320; prompt=v2/85343238; parser=ffb1d6e7; config=798f7eb1; malformed=11; malformed_rate=0.0127; truncated=6; noisy=0; invalid_items=0; unknown_label=1; unaligned_spans=32; git=f1d8cbd; torch=2.11.0+cu130; transformers=5.22.1; accelerate=1.15.0; bitsandbytes=0.50.2; seqeval=1.2.2
LLM-few-shot (Qwen2.5-7B-Instruct),none,hi,0,0,0.6713554987212277,0.5955631399317406,0.7189952904238618,0.6643391521197007,https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/tree/a09a35458c702b33eeacc393d103063234e8bc28,phase=5; mode=few-shot; k_shot=8

## MuRIL comparison model (Phase 3 handoff checkpoint)

The public handoff model at its pinned revision, run on the **same sentences** as the LLM so errors can be compared.
The LLM is already freed. On the full gold test set this must reproduce Phase 3's logged score (**0.8274**), which
checks this inference code.

In [13]:
MURIL_REPO = "priyadarshi-su/naamrekha-muril_hi_n300000_e3_lr2e-05_s42_bfd254ed"  # the Phase 3 handoff MuRIL checkpoint's Hugging Face repo
MURIL_REV = "de5204680ed4a8ab1ac842847df6866a6e2c9251"  # the exact pinned commit revision of that checkpoint

from transformers import AutoModelForTokenClassification, AutoTokenizer  # Hugging Face classes used to load the MuRIL checkpoint and its tokenizer
free_gpu()  # clear GPU memory left over from the LLM before loading MuRIL
m_tok = AutoTokenizer.from_pretrained(MURIL_REPO, revision=MURIL_REV)  # tokenizer pinned to the same commit as the model weights
m_model = AutoModelForTokenClassification.from_pretrained(MURIL_REPO, revision=MURIL_REV).to(device).eval()  # load the token-classification model and switch to inference mode
cfg_labels = {int(k): v for k, v in m_model.config.id2label.items()}  # the checkpoint's own saved label mapping, with string keys converted back to ints
## Why this guard exists here too (same pattern as in the error-analysis notebook): MuRIL's predicted label IDs are meaningless without knowing this checkpoint's own id-to-label mapping; without the check, a mismatched mapping would silently translate every predicted id into the WRONG tag string in predict_muril() rather than raising an error.
assert cfg_labels == id2label, f"checkpoint label order differs from the shared label map: {cfg_labels}"  # guard: a mismatched label order would silently scramble predicted tag ids

@torch.no_grad()  # disables gradient tracking: this is inference only
def predict_muril(rows, max_length=128, bs=32):  # runs MuRIL over pre-tokenized (word, gold_tags) rows and returns predicted tags
    preds = []  # collected predicted tag sequences
    for b in tqdm(range(0, len(rows), bs), desc="muril"):  # batched prediction with a progress bar
        batch = [w for w, _ in rows[b:b + bs]]  # this batch's word lists (gold tags aren't needed for prediction)
        enc = m_tok(batch, is_split_into_words=True, truncation=True, max_length=max_length, padding=True, return_tensors="pt")  # subword-tokenize the already word-split sentences
        ids = m_model(**{k: v.to(device) for k, v in enc.items()}).logits.argmax(-1).cpu().tolist()  # forward pass, take the highest-scoring label id per subword position
        for j, words in enumerate(batch):  # build the predicted tag sequence for each sentence in this batch
            tags, seen = ["O"] * len(words), set()  # default every word to "O"; `seen` tracks which words already got their label
            for pos, wid in enumerate(enc.word_ids(j)):  # walk each subword position and map it back to its original word
                if wid is not None and wid not in seen:          # first subword carries the word's label
                    seen.add(wid)  # mark this word as already labeled, so later subwords of the same word are ignored
                    tags[wid] = id2label[ids[j][pos]]  # translate the predicted label id into its string tag
            preds.append(tags)                                   # words lost to truncation stay O
    return preds  # list of predicted tag sequences, same order as `rows`

muril_pred = predict_muril(eval_rows)  # MuRIL's predictions on exactly the same sentences the LLM was just scored on
muril_scores = evaluate([t for _, t in eval_rows], muril_pred)  # seqeval entity-level scoring of those predictions against gold
print("MuRIL on", eval_name, {k: round(v, 4) for k, v in muril_scores.items()})  # prints MuRIL's scores for direct comparison against the LLM runs above
if STAGE == "final" and len(eval_rows) == len(test_all):  # only check reproduction when scoring the full, official gold test set
## Why re-derive Phase 3's score from scratch inside a DIFFERENT notebook instead of just trusting it: this notebook's MuRIL inference code (tokenization, batching, label alignment in predict_muril) is written independently of Phase 3's; reproducing the already-trusted 0.8274 number is what proves THIS notebook's own inference pipeline is correct, which the LLM-vs-MuRIL error comparison below depends on.
    assert abs(muril_scores["F1_overall"] - 0.8274) < 0.002, "MuRIL does not reproduce Phase 3's 0.8274: check inference code"  # confirms this notebook's MuRIL inference code matches Phase 3's, so the LLM-vs-MuRIL comparison is trustworthy
    print("reproduces the Phase 3 handoff score (0.8274)")  # confirms the check passed
del m_model, m_tok  # drop references so MuRIL's memory can be freed
free_gpu()  # clear GPU memory now that the comparison model is no longer needed

config.json:   0%|          | 0.00/984 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/6.41M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  948MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

muril:   0%|          | 0/27 [00:00<?, ?it/s]

MuRIL on gold test 864/864 {'F1_overall': 0.8274, 'F1_PER': 0.8879, 'F1_ORG': 0.7444, 'F1_LOC': 0.8224}
reproduces the Phase 3 handoff score (0.8274)


## Error analysis

Entity-level errors per system, compared with the gold spans. Every example shows **both** the gold span and what the
system predicted.

| Kind | Meaning |
| --- | --- |
| `missed` | a gold entity with no predicted span touching it |
| `boundary` | a predicted span overlaps the gold entity but with different boundaries (type may differ) |
| `type_confusion` | the exact gold span was predicted with another type |
| `spurious` | a predicted span that touches no gold entity |
| `format_drift` | malformed output (listed separately, not an entity error) |

In [14]:
from seqeval.metrics.sequence_labeling import get_entities  # turns a flat IOB2 tag sequence into a list of (type, start, end) entity spans

def spans(tags):  # convenience wrapper: get_entities returns (type, start, end) tuples; reorder to (start, end, type)
    return [(s, e, t) for t, s, e in get_entities(tags)]  # reorder seqeval's (type, start, end) into (start, end, type)

def classify(words, gold, pred):  # compares one sentence's gold vs predicted entities and labels each discrepancy with an error kind
    G, P = spans(gold), spans(pred)  # gold and predicted entity spans for this sentence
    gset, pset = set(G), set(P)  # sets for fast exact-match membership tests
    text = lambda sp: " ".join(words[sp[0]:sp[1] + 1])  # recovers the surface text of a span from the word list
    out = []  # accumulates one error record per mismatched gold or spurious predicted span
    for g in G:  # check every gold entity against what was predicted
        if g in pset:  # exact span and type match: no error to record
            continue  # this gold entity was predicted exactly right, so it's not an error
## Why check exact-boundary-wrong-type (same) before overlapping-different-boundary (over): an exact-boundary match also satisfies the overlap test, so checking `over` first would reclassify every type_confusion as a boundary error, hiding how often the model finds the right span but picks the wrong PER/ORG/LOC type.
        same = [p for p in P if (p[0], p[1]) == (g[0], g[1])]  # predicted spans with identical boundaries (so if present, only the type can be wrong)
        over = [p for p in P if (p[0], p[1]) != (g[0], g[1]) and not (p[1] < g[0] or p[0] > g[1])]  # predicted spans that overlap this gold entity but with different boundaries
        kind, p = ("type_confusion", same[0]) if same else ("boundary", over[0]) if over else ("missed", None)  # pick the matching error category and, if any, the offending prediction
        out.append(dict(kind=kind, gold=text(g), gold_type=g[2], pred=text(p) if p else "", pred_type=p[2] if p else ""))  # record both the gold and (if any) predicted text/type for this error
    for p in P:  # separately, find predicted spans that don't correspond to any gold entity at all
        if p not in gset and not any(not (g[1] < p[0] or g[0] > p[1]) for g in G):  # not an exact match and doesn't even overlap any gold entity
            out.append(dict(kind="spurious", gold="", gold_type="", pred=text(p), pred_type=p[2]))  # a prediction with nothing gold behind it at all
    return out  # list of per-entity error records for this sentence

systems = {f"LLM {mode} [{key}]": r["parsed"] for (key, mode), r in results.items()}  # every LLM (model, mode) run, keyed by a readable label
preds = {name: [p.tags for p in parsed] for name, parsed in systems.items()}  # just the recovered tag sequences, per system
preds["MuRIL (Phase 3)"] = muril_pred  # add the MuRIL comparison system alongside the LLM runs

err_rows = []  # flat list of per-entity error records across every system, feeding the `err` DataFrame
for name, ptags in preds.items():  # classify errors for every system being compared
    for i, ((w, g), p) in enumerate(zip(eval_rows, ptags)):  # pair each evaluated sentence with this system's prediction for it
        for e in classify(w, g, p):  # one or more error records for this sentence (possibly none, if everything was correct)
            err_rows.append(dict(system=name, idx=i, sentence=" ".join(w), **e))  # tag each error record with which system/sentence it came from, plus the full sentence for context
err = pd.DataFrame(err_rows)  # the full per-entity error table used by the rest of the notebook
print("error counts by kind:")  # section header for the table that follows
display(err.groupby(["system", "kind"]).size().unstack(fill_value=0))  # error-kind breakdown per system
print("by gold/predicted type (type = gold type, or predicted type for spurious):")  # section header for the table that follows
err["type"] = err.gold_type.where(err.gold_type != "", err.pred_type)  # for spurious errors (no gold_type), fall back to the predicted type so "type" is always meaningful
display(err.groupby(["system", "type", "kind"]).size().unstack(fill_value=0))  # same breakdown, further split by entity type

error counts by kind:


kind,boundary,missed,spurious,type_confusion
system,,,,
LLM few-shot [qwen2.5-7b-4bit],317,227,406,53
LLM zero-shot [qwen2.5-7b-4bit],293,299,478,52
MuRIL (Phase 3),167,77,205,31


by gold/predicted type (type = gold type, or predicted type for spurious):


kind                                  boundary  missed  spurious  \
system                          type                               
LLM few-shot [qwen2.5-7b-4bit]  LOC         83      49       121   
                                ORG         86      71       178   
                                PER        148     107       107   
LLM zero-shot [qwen2.5-7b-4bit] LOC         71      74       168   
                                ORG         79     107       175   
                                PER        143     118       135   
MuRIL (Phase 3)                 LOC         54      30        53   
                                ORG         64      37        94   
                                PER         49      10        58   

kind                                  type_confusion  
system                          type                  
LLM few-shot [qwen2.5-7b-4bit]  LOC               24  
                                ORG               19  
                                PER               10  
LLM zero-shot [qwen2.5-7b-4bit] LOC               16  
                                ORG               28  
                                PER                8  
MuRIL (Phase 3)                 LOC               16  
                                ORG                9  
                                PER                6

In [15]:
def show(system, mask, n=10, cols=("kind", "gold_type", "gold", "pred_type", "pred", "sentence")):  # prints a filtered slice of the error table for one system, with sentences truncated for readability
    sub = err[(err.system == system) & mask].head(n)  # the first n matching error rows for this system
    display(sub[list(cols)].assign(sentence=sub.sentence.str.slice(0, 90)))  # truncate long sentences to 90 characters so the table stays readable

_few = {f"LLM {m} [{k}]": r["scores"]["F1_overall"] for (k, m), r in results.items() if m == "few-shot"}  # F1 of every few-shot LLM run, keyed by display name
## Why pick ORG errors specifically to inspect first: the markdown above calls ORG "the weakest type across models" - the qualitative examples below are chosen to explain THAT specific, already-observed quantitative weakness, rather than inspecting errors at random.
best_llm = max(_few, key=_few.get)          # the few-shot run with the highest F1 on these sentences
print("=== ORG errors (the weakest type across models):", best_llm)  # section header naming which run is being inspected
show(best_llm, (err["type"] == "ORG"), n=12)  # qualitative look at the best LLM's organization-name errors
print("=== ORG errors: MuRIL")  # section header for the MuRIL comparison
show("MuRIL (Phase 3)", (err["type"] == "ORG"), n=12)  # same, for the MuRIL comparison model
for kind in ["boundary", "missed", "spurious", "type_confusion"]:  # show a few examples of each error kind in turn
    print(f"\n=== {kind}: {best_llm}")  # section header naming this error kind
    show(best_llm, (err.kind == kind), n=8)  # qualitative examples of each error kind for the best LLM run

=== ORG errors (the weakest type across models): LLM few-shot [qwen2.5-7b-4bit]


,kind,gold_type,gold,pred_type,pred,sentence
1124,spurious,,,ORG,सरकार,"सरकार ने नवम्बर , 2007 में सिंगापुर में आसियान..."
1125,spurious,,,ORG,लुक ईस्ट,लुक ईस्ट पॉलिसी
1127,missed,ORG,राष्ट्रीय स्वच्छ विद्यालय,,,"चार राज्यों ( पुड्डुचेरी , तमिलनाडु , गुजरात औ..."
1128,spurious,,,ORG,राज्य मंत्रिमंडल,यह फैसला मुख्यमंत्री वीरभद्र सिंह की अध्यक्षता...
1132,boundary,ORG,केंद्रीय पर्यावरण,ORG,केंद्रीय पर्यावरण मंत्री एवं वन मंत्रालय,सुल्ताना ने कहा कि पंजाब सरकार को बड़ी सड़क पर...
1133,boundary,ORG,वन मंत्रालय,ORG,केंद्रीय पर्यावरण मंत्री एवं वन मंत्रालय,सुल्ताना ने कहा कि पंजाब सरकार को बड़ी सड़क पर...
1136,spurious,,,ORG,इंडिया हैकथन,विश्व की सबसे बड़ी राष्ट्र निर्माण डिजिटल पहल ...
1137,type_confusion,ORG,पंजाब राज्य,LOC,पंजाब राज्य,"646 बचन सिंह बनाम पंजाब राज्य , ( 1982 ) 3 एसस..."
1138,missed,ORG,एससीसी,,,"646 बचन सिंह बनाम पंजाब राज्य , ( 1982 ) 3 एसस..."
1139,boundary,ORG,अखिल भारतीय कांग्रेस,ORG,अखिल भारतीय कांग्रेस पार्टी,"राज्यपाल से बैठक के बाद कुमारस्वामी ने कहा , ""..."


=== ORG errors: MuRIL


,kind,gold_type,gold,pred_type,pred,sentence
2128,spurious,,,ORG,लुक ईस्ट,लुक ईस्ट पॉलिसी
2130,missed,ORG,राष्ट्रीय स्वच्छ विद्यालय,,,"चार राज्यों ( पुड्डुचेरी , तमिलनाडु , गुजरात औ..."
2131,missed,ORG,केंद्रीय पर्यावरण,,,सुल्ताना ने कहा कि पंजाब सरकार को बड़ी सड़क पर...
2132,boundary,ORG,वन मंत्रालय,ORG,मंत्रालय,सुल्ताना ने कहा कि पंजाब सरकार को बड़ी सड़क पर...
2135,boundary,ORG,अखिल भारतीय कांग्रेस,ORG,अखिल भारतीय कांग्रेस पार्टी,"राज्यपाल से बैठक के बाद कुमारस्वामी ने कहा , ""..."
2136,type_confusion,ORG,सिंदरी,LOC,सिंदरी,"सिफारिशों के अनुसार एफसीआईएल की सिंदरी , तलचर ..."
2137,boundary,ORG,पीएसयू,ORG,पीएसयू के परिसंघ,"सिफारिशों के अनुसार एफसीआईएल की सिंदरी , तलचर ..."
2140,spurious,,,ORG,सौर जल,सौर जल - पम्प
2141,missed,ORG,भारतीय रक्षा कंपनियों,,,वेबिनार के दौरान 12 भारतीय रक्षा कंपनियों - अश...
2142,missed,ORG,बॉक्स,,,वह बॉक्स में और संगठन के बाहर भी तकनीक में महि...



=== boundary: LLM few-shot [qwen2.5-7b-4bit]


,kind,gold_type,gold,pred_type,pred,sentence
1129,boundary,PER,मोहम्मदVI,PER,महामहिम किंग मोहम्मदVI,"भारत के राष्ट्रपति , श्री प्रणब मुखर्जी ने महा..."
1132,boundary,ORG,केंद्रीय पर्यावरण,ORG,केंद्रीय पर्यावरण मंत्री एवं वन मंत्रालय,सुल्ताना ने कहा कि पंजाब सरकार को बड़ी सड़क पर...
1133,boundary,ORG,वन मंत्रालय,ORG,केंद्रीय पर्यावरण मंत्री एवं वन मंत्रालय,सुल्ताना ने कहा कि पंजाब सरकार को बड़ी सड़क पर...
1134,boundary,PER,श्री,PER,श्री प्रकाश जावड़ेकर,विश्व की सबसे बड़ी राष्ट्र निर्माण डिजिटल पहल ...
1135,boundary,PER,प्रकाश जावड़ेकर,PER,श्री प्रकाश जावड़ेकर,विश्व की सबसे बड़ी राष्ट्र निर्माण डिजिटल पहल ...
1139,boundary,ORG,अखिल भारतीय कांग्रेस,ORG,अखिल भारतीय कांग्रेस पार्टी,"राज्यपाल से बैठक के बाद कुमारस्वामी ने कहा , ""..."
1140,boundary,ORG,कांग्रेस,ORG,कांग्रेस नेताओं,"राज्यपाल से बैठक के बाद कुमारस्वामी ने कहा , ""..."
1146,boundary,LOC,हरियाणा,LOC,हरियाणा राज्य,सिवानी ( Siwani ) भारत के हरियाणा राज्य के भिव...


... (2 outputs left out)


=== spurious: LLM few-shot [qwen2.5-7b-4bit]


,kind,gold_type,gold,pred_type,pred,sentence
1122,spurious,,,LOC,2017,गुजरात हाईकोर्ट ने मंगलवार को कांग्रेस नेता अह...
1124,spurious,,,ORG,सरकार,"सरकार ने नवम्बर , 2007 में सिंगापुर में आसियान..."
1125,spurious,,,ORG,लुक ईस्ट,लुक ईस्ट पॉलिसी
1126,spurious,,,PER,के,के कार्यान्वयन में उल्लेखनीय प्रगति की है ।
1128,spurious,,,ORG,राज्य मंत्रिमंडल,यह फैसला मुख्यमंत्री वीरभद्र सिंह की अध्यक्षता...
1130,spurious,,,PER,राष्ट्रपति,"भारत के राष्ट्रपति , श्री प्रणब मुखर्जी ने महा..."
1131,spurious,,,PER,किंग,"भारत के राष्ट्रपति , श्री प्रणब मुखर्जी ने महा..."
1136,spurious,,,ORG,इंडिया हैकथन,विश्व की सबसे बड़ी राष्ट्र निर्माण डिजिटल पहल ...



=== type_confusion: LLM few-shot [qwen2.5-7b-4bit]


,kind,gold_type,gold,pred_type,pred,sentence
1123,type_confusion,LOC,आसियान,ORG,आसियान,"सरकार ने नवम्बर , 2007 में सिंगापुर में आसियान..."
1137,type_confusion,ORG,पंजाब राज्य,LOC,पंजाब राज्य,"646 बचन सिंह बनाम पंजाब राज्य , ( 1982 ) 3 एसस..."
1144,type_confusion,ORG,सिंदरी,LOC,सिंदरी,"सिफारिशों के अनुसार एफसीआईएल की सिंदरी , तलचर ..."
1151,type_confusion,ORG,नेस्डैक,LOC,नेस्डैक,नेस्डैक में शेयरों के उछाल की अनुक्रिया भारत म...
1233,type_confusion,LOC,6 सीजीओ काम्प्लेक्स,ORG,6 सीजीओ काम्प्लेक्स,"इलेक्ट्रॉ निक्सव निकेतन , 6 सीजीओ काम्प्लेक्स ..."
1266,type_confusion,ORG,जम्मू - कश्मीर,LOC,जम्मू - कश्मीर,भाजपा नेता ने कहा कि विदेशी प्रतिनिधिमंडल को अ...
1277,type_confusion,LOC,आसियान,ORG,आसियान,चीन के बाद आसियान भारत का दूसरा सबसे बड़ा व्या...
1289,type_confusion,LOC,सिविल अस्पताल,ORG,सिविल अस्पताल,चारों को सिविल अस्पताल में भर्ती करवाया गया है ।


In [16]:
# Format drift and unaligned spans, per LLM run.
for (key, mode), r in results.items():  # print full diagnostics for every model/mode run
    st = r["stats"]  # this run's diagnostics dict
    print(f"\n=== {key} / {mode}: malformed {st['malformed']}/{st['n']} {st['malformed_reasons']} | "  # banner line (continued below): identifies the run and its malformed-answer count
          f"truncated {st['truncated']} | with prose or fence {st['noisy']} | invalid items {st['invalid_items']} | "  # (continued) more failure-mode counts
          f"unknown labels {st['unknown_label']} | unaligned spans {st['unaligned']} | overlap-dropped {st['overlap_dropped']}")  # full diagnostics for one model/mode run
    bad = [i for i, p in enumerate(r["parsed"]) if p.malformed]  # indices of sentences whose answer the parser could not interpret at all
    for i in bad[:6]:  # show at most 6 example malformed outputs, to avoid flooding the notebook
        print(f"  malformed [{r['parsed'][i].reason}] ->", repr(r["raw"][i][:160]))  # show a few raw (truncated) model outputs that failed to parse, for manual inspection
## Why tally unaligned entity texts specifically (not just count them): a text that is CONSISTENTLY unaligned across many sentences (e.g. the model translating a common name) points to a systematic parser/prompt issue worth fixing; a few one-off unaligned texts are more likely ordinary model mistakes - the distinction only shows up by looking at the most_common() breakdown, not the raw count.
    un = Counter(t for p in r["parsed"] for t in p.unaligned_texts)  # tally of entity texts the model wrote that don't actually appear in the sentence's words
    print("  most common unaligned spans (text the model wrote that is not in the sentence):", un.most_common(10))  # the most frequent such hallucinated/misaligned spans

# Sentences where MuRIL is exactly right and the LLM is not, and the reverse.
g_all = [t for _, t in eval_rows]  # the gold tags for every evaluated sentence
muril_ok = [p == g for p, g in zip(muril_pred, g_all)]  # per-sentence: did MuRIL get the entire tag sequence exactly right
for name in [s for s in preds if s.startswith("LLM")]:  # compare each LLM run against MuRIL in turn
    llm_ok = [p == g for p, g in zip(preds[name], g_all)]  # per-sentence: did this LLM run get the entire tag sequence exactly right
    print(f"{name}: both exact {sum(a and b for a, b in zip(muril_ok, llm_ok))} | MuRIL only {sum(a and not b for a, b in zip(muril_ok, llm_ok))} "  # four-way exact-match agreement breakdown for this LLM run vs MuRIL
          f"| LLM only {sum(b and not a for a, b in zip(muril_ok, llm_ok))} | neither {sum(not a and not b for a, b in zip(muril_ok, llm_ok))}")  # four-way breakdown of per-sentence exact-match agreement between MuRIL and this LLM run


=== qwen2.5-7b-4bit / zero-shot: malformed 11/864 {'broken_json': 11} | truncated 6 | with prose or fence 0 | invalid items 0 | unknown labels 1 | unaligned spans 32 | overlap-dropped 4
  malformed [broken_json] -> '[{"text": "अशोक लीलैंड", "type": "ORG"}, {"text": "बीईएमएल", "type": "ORG"}, {"text": "भारत इलेक्ट्रोनिक्स लिमिटेड", "type": "ORG"}, {"text": "हिन्दुस्तान एयरोन'
  malformed [broken_json] -> '[{"text": "इलेक्ट्रॉ निक्सव निकेतन", "type": "ORG"}, {"text": "6 सीजीओ काम्प्लेक्स", "type": "ORG"}, {"text": "लोदी रोड", "type": "LOC"]}'
  malformed [broken_json] -> '[{"text": "पर्थ स्कॉर्चर्स", "type": "ORG"}, {"text": "कप्तान", "type": "PER"}, {"text": "मिचेल मार्श", "type": "PER"}, {"text": "जेसन रॉय", "type": "PER"}, {"t'
  malformed [broken_json] -> '[{"text": "साउथ अफ्रीका", "type": "LOC"}, {"text": "फाफ डु प्लेसिस", "type": "PER"}, {"text": "डील एगर", "type": "PER"}, {"text": "एडेन मार्कराम", "type": "PER"'
  malformed [broken_json] -> '[{"text": "पानीपत", "type": "LOC"}, {"

## Save per-sentence artifacts

Raw model output, parse flags and predicted tags for every sentence and system, plus the error table. These back every
claim in `docs/phase5/` (final stage: download them from the Output tab and commit unchanged).

In [17]:
pred_path = os.path.join(ART_DIR, f"{STAGE}_predictions.jsonl")  # output path for every system's per-sentence predictions
with open(pred_path, "w", encoding="utf-8") as f:  # write every system's per-sentence predictions as JSON Lines
    for name, ptags in preds.items():  # one system at a time
        llm = next(((k, m) for (k, m) in results if f"LLM {m} [{k}]" == name), None)  # find this system's (model key, mode) if it's an LLM run, else None for MuRIL
        for i, ((w, g), p) in enumerate(zip(eval_rows, ptags)):  # one sentence at a time for this system
            rec = dict(system=name, idx=i, words=w, gold=g, pred=p)  # base record: this system's prediction for this sentence
            if llm:  # this system is an LLM run, so it has raw text and parser diagnostics to attach
                pr = results[llm]["parsed"][i]  # the parser's full result object for this sentence, including malformed/diagnostic flags
                rec.update(raw=results[llm]["raw"][i], malformed=pr.malformed, reason=pr.reason,  # attach LLM-only diagnostics (continued below)
                           unaligned=pr.unaligned_texts, truncated=results[llm]["truncated"][i])  # add LLM-only diagnostics: the raw text and why/whether parsing struggled
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")  # one JSON line per (system, sentence) record
err_path = os.path.join(ART_DIR, f"{STAGE}_errors.csv")  # output path for the full per-entity error table
err.to_csv(err_path, index=False, encoding="utf-8")  # write the error table so every claim in the docs can be traced back to it
print("wrote", pred_path, "and", err_path)  # confirms both artifact files were written successfully

wrote /kaggle/working/NaamRekha/results/phase5/final_predictions.jsonl and /kaggle/working/NaamRekha/results/phase5/final_errors.csv


## Comparison with the other stages (from `results.csv`)

Read from the committed results sheet, not retyped. Phase 4 rows are not used.

In [18]:
import csv  # used to read results/results.csv as dict rows
from src import RESULTS_PATH  # the shared scoreboard's file path
with open(RESULTS_PATH, encoding="utf-8", newline="") as f:  # newline="" is required so csv's own universal-newline handling isn't double-applied
    sheet = list(csv.DictReader(f))  # the whole shared scoreboard, as a list of {column: value} dicts
want = [r for r in sheet if r["test_lang"] == "hi" and (r["model"].startswith("LLM-")  # filter down to Hindi rows for the relevant systems (continued below)
        or r["model"] == "BiLSTM-CRF" or (r["model"] in ("MuRIL", "IndicBERT") and r["subset_size"] == "300000" and "_s42_" in r["checkpoint_path"]))]  # pick out exactly the Hindi rows relevant to this comparison: both LLM modes, the BiLSTM-CRF baseline, and the seed-42/300k MuRIL and IndicBERT runs
cols = ("model", "subset_size", "F1_PER", "F1_ORG", "F1_LOC", "F1_overall")  # the columns shown in the comparison table
table, seen = [], set()  # `seen` dedupes rows that appear more than once in the sheet
for r in want:                       # Phase 4 re-scored the Phase 3 Hindi models: show each distinct result once
## Why dedupe by a rounded-score tuple instead of by row index or model name alone: Phase 4 re-scored some Phase 3 Hindi models and logged them again under the same model/checkpoint identity; rounding scores before building the dedup key collapses truly-identical repeated results while still keeping genuinely different (re-scored, slightly different) rows distinct.
    key = tuple(round(float(r[k]), 4) if k.startswith("F1") else r[k] for k in cols)  # a dedup key: rounded scores plus the non-numeric columns
    if key not in seen:  # only keep the first occurrence of each distinct result
        seen.add(key)  # mark this result as already shown
        table.append(dict(zip(cols, key)))  # keep only the first occurrence of each distinct result
display(pd.DataFrame(table))  # the final side-by-side comparison: BiLSTM-CRF, MuRIL, IndicBERT, and both LLM modes, all on Hindi

,model,subset_size,F1_PER,F1_ORG,F1_LOC,F1_overall
0,BiLSTM-CRF,100000,0.8692,0.7130,0.7933,0.8038
1,MuRIL,300000,0.8879,0.7444,0.8224,0.8274
2,IndicBERT,300000,0.8538,0.6968,0.7692,0.7839
3,LLM-zero-shot (Qwen2.5-7B-Instruct),0,0.6397,0.5644,0.6812,0.6328
4,LLM-few-shot (Qwen2.5-7B-Instruct),0,0.6714,0.5956,0.7190,0.6643


## Findings

Written after the run, from the printed outputs above. Leave this empty until then; the final write-up lives in
`docs/phase5/README.md` and `docs/phase5/error_analysis.md`.